## Plot Generation - Effect Size Analysis

### Cliff's Delta

In [20]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle


# ============================================================
# GLOBAL STYLE
# ============================================================

# Set to True for the closest match to the LaTeX-style reference.
# This requires a working LaTeX installation.
USE_LATEX = True

if USE_LATEX:
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman"],
        "font.size": 8,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.pad_inches": 0.01,
    })
else:
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": [
            "CMU Serif",
            "Computer Modern Roman",
            "DejaVu Serif",
        ],
        "font.size": 8,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.pad_inches": 0.01,
    })


# ============================================================
# TOOL LABELS
# ============================================================

DEFAULT_DISPLAY_NAMES = {
    "autogen": "AutoGen",
    "autogpt": "AutoGPT",
    "dify": "Dify",
    "semantic_kernel": "Sem. Kernel",
    "semantic_kernel_chat": "SKC",
}

DEFAULT_ABBREVIATIONS = {
    "autogen": "AGE",
    "autogpt": "AGP",
    "dify": "DIF",
    "semantic_kernel": "SEK",
    "semantic_kernel_chat": "SKC",
}


# ============================================================
# COLORS
# ============================================================

def effect_background(effect_size):
    """
    Background depends only on the effect_size column.

    The same effect-size category always receives the same color,
    regardless of whether Cliff's delta is positive or negative.
    """

    colors = {
        "negligible": "#D9EAD3",  # light green
        "small": "#FFF2CC",       # light yellow
        "medium": "#FCE5CD",      # light orange
        "large": "#F4CCCC",       # light red
    }

    normalized_effect = str(effect_size).strip().lower()

    return colors.get(
        normalized_effect,
        "#FFFFFF",
    )


# ============================================================
# MATRIX BRACKETS
# ============================================================

def draw_bracket(
    ax,
    x,
    y_bottom,
    y_top,
    side,
):
    """
    Draw a thin square bracket around the matrix.
    """

    arm_length = 0.09
    line_width = 0.7

    ax.plot(
        [x, x],
        [y_bottom, y_top],
        color="black",
        linewidth=line_width,
        clip_on=False,
    )

    if side == "left":

        ax.plot(
            [x, x + arm_length],
            [y_top, y_top],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

        ax.plot(
            [x, x + arm_length],
            [y_bottom, y_bottom],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

    elif side == "right":

        ax.plot(
            [x - arm_length, x],
            [y_top, y_top],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

        ax.plot(
            [x - arm_length, x],
            [y_bottom, y_bottom],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

    else:
        raise ValueError(
            "side must be either 'left' or 'right'."
        )


# ============================================================
# FILE UTILITIES
# ============================================================

def safe_filename(value):
    """
    Convert a metric name into a filename-safe string.
    """

    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
    )


# ============================================================
# VALUE FORMATTING
# ============================================================

def format_delta(
    delta,
    decimals=3,
):
    """
    Format Cliff's delta using only its sign.

    Examples
    --------
    +0.032
    -0.032
     0.000
    """

    if delta == 0:
        return f"{delta:.{decimals}f}"

    return f"{delta:+.{decimals}f}"


# ============================================================
# PDF GENERATOR
# ============================================================

def generate_cliffs_delta_pdfs(
    csv_path,
    output_directory="cliffs_delta_matrices",
    metric_order=(
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ),
    tool_order=None,
    display_names=None,
    abbreviations=None,
    decimals=3,
    show_effect_size=False,
):
    """
    Generate one separate PDF per metric.

    Expected CSV columns
    --------------------
    metric
    tool_1
    tool_2
    cliffs_delta
    effect_size

    Matrix orientation
    ------------------
    Columns correspond to tool_1.
    Rows correspond to tool_2.

    Up arrow:
        the column tool tends to have higher values.

    Down arrow:
        the column tool tends to have lower values.
    """

    csv_path = Path(csv_path)
    output_directory = Path(output_directory)

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV file not found: {csv_path}"
        )

    df = pd.read_csv(csv_path)

    required_columns = {
        "metric",
        "tool_1",
        "tool_2",
        "cliffs_delta",
        "effect_size",
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            "CSV is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    df["metric"] = (
        df["metric"]
        .astype(str)
        .str.strip()
    )

    df["tool_1"] = (
        df["tool_1"]
        .astype(str)
        .str.strip()
    )

    df["tool_2"] = (
        df["tool_2"]
        .astype(str)
        .str.strip()
    )

    df["effect_size"] = (
        df["effect_size"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["cliffs_delta"] = pd.to_numeric(
        df["cliffs_delta"],
        errors="raise",
    )

    display_names = {
        **DEFAULT_DISPLAY_NAMES,
        **(display_names or {}),
    }

    abbreviations = {
        **DEFAULT_ABBREVIATIONS,
        **(abbreviations or {}),
    }

    discovered_tools = list(
        dict.fromkeys(
            df["tool_1"].tolist()
            + df["tool_2"].tolist()
        )
    )

    if tool_order is None:
        tools = discovered_tools
    else:
        tools = list(tool_order)

    available_metrics = list(
        dict.fromkeys(
            df["metric"].tolist()
        )
    )

    if metric_order is None:
        metrics = available_metrics
    else:
        metrics = [
            metric
            for metric in metric_order
            if metric in available_metrics
        ]

        metrics.extend(
            metric
            for metric in available_metrics
            if metric not in metrics
        )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated_files = []

    for metric in metrics:

        metric_df = df[
            df["metric"] == metric
        ].copy()

        lookup = {
            (
                row.tool_1,
                row.tool_2,
            ): row
            for row in metric_df.itertuples(
                index=False
            )
        }

        number_of_tools = len(tools)

        # Approximate dimensions of the uploaded reference PDF.
        figure_width = 222.71 / 72
        figure_height = 150.70 / 72

        fig, ax = plt.subplots(
            figsize=(
                figure_width,
                figure_height,
            )
        )

        ax.set_xlim(
            -0.70,
            number_of_tools + 0.20,
        )

        ax.set_ylim(
            number_of_tools + 0.18,
            -1.22,
        )

        ax.axis("off")

        fig.subplots_adjust(
            left=0.02,
            right=0.99,
            bottom=0.02,
            top=0.99,
        )

        # ----------------------------------------------------
        # COLUMN LABELS
        # ----------------------------------------------------

        for column_index, tool in enumerate(tools):

            ax.text(
                column_index + 0.43,
                -0.25,
                display_names.get(
                    tool,
                    tool,
                ),
                rotation=48,
                rotation_mode="anchor",
                ha="left",
                va="bottom",
                fontsize=8,
                fontweight="normal",
                color="black",
            )

        # ----------------------------------------------------
        # ROWS AND CELLS
        # ----------------------------------------------------

        for row_index, row_tool in enumerate(tools):

            ax.text(
                -0.24,
                row_index + 0.50,
                abbreviations.get(
                    row_tool,
                    row_tool,
                ),
                ha="right",
                va="center",
                fontsize=8,
                fontweight="normal",
                color="black",
            )

            for (
                column_index,
                column_tool,
            ) in enumerate(tools):

                x_position = column_index + 0.50
                y_position = row_index + 0.50

                # Diagonal and upper triangle.
                if column_index >= row_index:

                    ax.text(
                        x_position,
                        y_position,
                        "--",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="black",
                        fontweight="normal",
                    )

                    continue

                # Preferred orientation:
                # tool_1 = column
                # tool_2 = row
                record = lookup.get(
                    (
                        column_tool,
                        row_tool,
                    )
                )

                sign_multiplier = 1.0

                # If only the reverse comparison exists,
                # invert the sign.
                if record is None:

                    record = lookup.get(
                        (
                            row_tool,
                            column_tool,
                        )
                    )

                    sign_multiplier = -1.0

                if record is None:

                    ax.text(
                        x_position,
                        y_position,
                        "NA",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="black",
                    )

                    continue

                delta = (
                    float(record.cliffs_delta)
                    * sign_multiplier
                )

                magnitude = (
                    str(record.effect_size)
                    .strip()
                    .lower()
                )

                # Background encodes effect-size magnitude only.
                ax.add_patch(
                    Rectangle(
                        (
                            column_index + 0.055,
                            row_index + 0.055,
                        ),
                        0.89,
                        0.89,
                        facecolor=effect_background(
                            magnitude
                        ),
                        edgecolor="none",
                        linewidth=0,
                        zorder=1,
                    )
                )

                value_text = format_delta(
                    delta=delta,
                    decimals=decimals,
                )

                if show_effect_size:
                    if USE_LATEX:
                        value_text += (
                            rf"\newline "
                            rf"\scriptsize{{({magnitude})}}"
                        )
                    else:
                        value_text += f"\n({magnitude})"

                ax.text(
                    x_position,
                    y_position,
                    value_text,
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="normal",
                    color="black",
                    zorder=2,
                )

        # ----------------------------------------------------
        # BRACKETS
        # ----------------------------------------------------

        draw_bracket(
            ax=ax,
            x=-0.03,
            y_bottom=0.12,
            y_top=number_of_tools - 0.12,
            side="left",
        )

        draw_bracket(
            ax=ax,
            x=number_of_tools + 0.03,
            y_bottom=0.12,
            y_top=number_of_tools - 0.12,
            side="right",
        )

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        metric_filename = (
            f"cliffs_delta_"
            f"{safe_filename(metric)}.pdf"
        )

        metric_output_path = (
            output_directory
            / metric_filename
        )

        fig.savefig(
            metric_output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.01,
            transparent=False,
        )

        plt.close(fig)

        generated_files.append(
            metric_output_path
        )

        print(
            f"Created: {metric_output_path}"
        )

    return generated_files




In [22]:
generated_files = generate_cliffs_delta_pdfs(
    csv_path="cliffsDelta_50.csv",
    output_directory="cliffs_delta_matrices",

    metric_order=(
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ),

    tool_order=(
        "autogen",
        "autogpt",
        "dify",
        "semantic_kernel",
        "semantic_kernel_chat",
    ),

    decimals=3,

    # Set to True to include labels such as "(negligible)"
    # below each number.
    show_effect_size=False,
)

print("\nGenerated files:")

for file_path in generated_files:
    print(file_path)

Created: cliffs_delta_matrices/cliffs_delta_ROUGE-1.pdf
Created: cliffs_delta_matrices/cliffs_delta_ROUGE-2.pdf
Created: cliffs_delta_matrices/cliffs_delta_ROUGE-L.pdf

Generated files:
cliffs_delta_matrices/cliffs_delta_ROUGE-1.pdf
cliffs_delta_matrices/cliffs_delta_ROUGE-2.pdf
cliffs_delta_matrices/cliffs_delta_ROUGE-L.pdf


### Rank-biserial Correlation

In [23]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle


# ============================================================
# GLOBAL STYLE
# ============================================================

# Set to True for the closest match to the LaTeX-style example.
# This requires a working LaTeX installation.
USE_LATEX = True

if USE_LATEX:
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman"],
        "font.size": 8,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.pad_inches": 0.01,
    })
else:
    # Fallback when LaTeX is unavailable.
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": [
            "CMU Serif",
            "Computer Modern Roman",
            "DejaVu Serif",
        ],
        "font.size": 8,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.pad_inches": 0.01,
    })


# ============================================================
# TOOL LABELS
# ============================================================

DEFAULT_DISPLAY_NAMES = {
    "autogen": "AutoGen",
    "autogpt": "AutoGPT",
    "dify": "Dify",
    "semantic_kernel": "Sem. Kernel",
    "semantic_kernel_chat": "SKC",
}

DEFAULT_ABBREVIATIONS = {
    "autogen": "AGE",
    "autogpt": "AGP",
    "dify": "DIF",
    "semantic_kernel": "SEK",
    "semantic_kernel_chat": "SKC",
}


# ============================================================
# EFFECT-SIZE COLORS
# ============================================================

def magnitude_background(magnitude):
    """
    Return the background color associated with the magnitude.

    The background represents only effect-size magnitude.
    The sign of rank-biserial correlation does not affect color.
    """

    colors = {
        "negligible": "#D9EAD3",  # light green
        "small": "#FFF2CC",       # light yellow
        "medium": "#FCE5CD",      # light orange
        "large": "#F4CCCC",       # light red
    }

    normalized_magnitude = str(magnitude).strip().lower()

    return colors.get(
        normalized_magnitude,
        "#FFFFFF",
    )


# ============================================================
# VALUE FORMATTING
# ============================================================

def format_rank_biserial(
    value,
    decimals=3,
):
    """
    Format rank-biserial correlation with an explicit sign.

    Examples
    --------
    +0.231
    -0.231
     0.000
    """

    # Avoid displaying -0.000 because of floating-point rounding.
    rounding_threshold = 0.5 * (10 ** -decimals)

    if abs(value) < rounding_threshold:
        value = 0.0

    if value == 0:
        return f"{value:.{decimals}f}"

    return f"{value:+.{decimals}f}"


# ============================================================
# MATRIX BRACKETS
# ============================================================

def draw_bracket(
    ax,
    x,
    y_bottom,
    y_top,
    side,
):
    """
    Draw a thin square bracket around the matrix.
    """

    arm_length = 0.09
    line_width = 0.7

    ax.plot(
        [x, x],
        [y_bottom, y_top],
        color="black",
        linewidth=line_width,
        clip_on=False,
    )

    if side == "left":
        ax.plot(
            [x, x + arm_length],
            [y_top, y_top],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

        ax.plot(
            [x, x + arm_length],
            [y_bottom, y_bottom],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

    elif side == "right":
        ax.plot(
            [x - arm_length, x],
            [y_top, y_top],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

        ax.plot(
            [x - arm_length, x],
            [y_bottom, y_bottom],
            color="black",
            linewidth=line_width,
            clip_on=False,
        )

    else:
        raise ValueError(
            "side must be either 'left' or 'right'."
        )


# ============================================================
# FILE UTILITIES
# ============================================================

def safe_filename(value):
    """
    Convert a metric name into a filename-safe string.
    """

    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
    )


# ============================================================
# PDF GENERATOR
# ============================================================

def generate_rank_biserial_pdfs(
    csv_path,
    output_directory="rank_biserial_matrices",
    metric_order=(
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ),
    tool_order=None,
    display_names=None,
    abbreviations=None,
    decimals=3,
    show_magnitude=False,
):
    """
    Generate one separate rank-biserial matrix PDF per metric.

    Expected CSV columns
    --------------------
    metric
    tool_1
    tool_2
    rank_biserial
    magnitude

    Optional CSV columns
    --------------------
    direction
    n_pairs

    Matrix orientation
    ------------------
    Columns correspond to tool_1.
    Rows correspond to tool_2.

    Positive value:
        the column tool tends to have larger values than the row tool.

    Negative value:
        the column tool tends to have smaller values than the row tool.
    """

    csv_path = Path(csv_path)
    output_directory = Path(output_directory)

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV file not found: {csv_path}"
        )

    df = pd.read_csv(csv_path)

    required_columns = {
        "metric",
        "tool_1",
        "tool_2",
        "rank_biserial",
        "magnitude",
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            "CSV is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # --------------------------------------------------------
    # CLEAN DATA
    # --------------------------------------------------------

    df["metric"] = (
        df["metric"]
        .astype(str)
        .str.strip()
    )

    df["tool_1"] = (
        df["tool_1"]
        .astype(str)
        .str.strip()
    )

    df["tool_2"] = (
        df["tool_2"]
        .astype(str)
        .str.strip()
    )

    df["magnitude"] = (
        df["magnitude"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["rank_biserial"] = pd.to_numeric(
        df["rank_biserial"],
        errors="raise",
    )

    # Check for duplicated comparisons within each metric.
    duplicate_mask = df.duplicated(
        subset=[
            "metric",
            "tool_1",
            "tool_2",
        ],
        keep=False,
    )

    if duplicate_mask.any():
        duplicates = df.loc[
            duplicate_mask,
            [
                "metric",
                "tool_1",
                "tool_2",
            ],
        ]

        raise ValueError(
            "Duplicated comparisons were found:\n"
            f"{duplicates.to_string(index=False)}"
        )

    display_names = {
        **DEFAULT_DISPLAY_NAMES,
        **(display_names or {}),
    }

    abbreviations = {
        **DEFAULT_ABBREVIATIONS,
        **(abbreviations or {}),
    }

    # Preserve the order in which tools first appear.
    discovered_tools = list(
        dict.fromkeys(
            df["tool_1"].tolist()
            + df["tool_2"].tolist()
        )
    )

    if tool_order is None:
        tools = discovered_tools
    else:
        tools = list(tool_order)

        unknown_tools = [
            tool
            for tool in tools
            if tool not in discovered_tools
        ]

        if unknown_tools:
            raise ValueError(
                "The following tools in tool_order were not found "
                f"in the CSV: {unknown_tools}"
            )

    available_metrics = list(
        dict.fromkeys(
            df["metric"].tolist()
        )
    )

    if metric_order is None:
        metrics = available_metrics
    else:
        metrics = [
            metric
            for metric in metric_order
            if metric in available_metrics
        ]

        # Add metrics not listed in metric_order.
        metrics.extend(
            metric
            for metric in available_metrics
            if metric not in metrics
        )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated_files = []

    # ========================================================
    # CREATE ONE PDF PER METRIC
    # ========================================================

    for metric in metrics:

        metric_df = df[
            df["metric"] == metric
        ].copy()

        lookup = {
            (
                row.tool_1,
                row.tool_2,
            ): row
            for row in metric_df.itertuples(
                index=False
            )
        }

        number_of_tools = len(tools)

        # Approximate dimensions of the reference PDF:
        # 222.71 × 150.70 PDF points.
        # 72 PDF points = 1 inch.
        figure_width = 222.71 / 72
        figure_height = 150.70 / 72

        fig, ax = plt.subplots(
            figsize=(
                figure_width,
                figure_height,
            )
        )

        ax.set_xlim(
            -0.70,
            number_of_tools + 0.20,
        )

        ax.set_ylim(
            number_of_tools + 0.18,
            -1.22,
        )

        ax.axis("off")

        fig.subplots_adjust(
            left=0.02,
            right=0.99,
            bottom=0.02,
            top=0.99,
        )

        # ----------------------------------------------------
        # COLUMN LABELS
        # ----------------------------------------------------

        for column_index, tool in enumerate(tools):

            ax.text(
                column_index + 0.43,
                -0.25,
                display_names.get(
                    tool,
                    tool,
                ),
                rotation=48,
                rotation_mode="anchor",
                ha="left",
                va="bottom",
                fontsize=8,
                fontweight="normal",
                color="black",
            )

        # ----------------------------------------------------
        # MATRIX ROWS AND CELLS
        # ----------------------------------------------------

        for row_index, row_tool in enumerate(tools):

            ax.text(
                -0.24,
                row_index + 0.50,
                abbreviations.get(
                    row_tool,
                    row_tool,
                ),
                ha="right",
                va="center",
                fontsize=8,
                fontweight="normal",
                color="black",
            )

            for column_index, column_tool in enumerate(tools):

                x_position = column_index + 0.50
                y_position = row_index + 0.50

                # Diagonal and upper triangle.
                if column_index >= row_index:
                    ax.text(
                        x_position,
                        y_position,
                        "--",
                        ha="center",
                        va="center",
                        fontsize=8,
                        fontweight="normal",
                        color="black",
                    )

                    continue

                # Preferred orientation:
                # tool_1 = column
                # tool_2 = row
                record = lookup.get(
                    (
                        column_tool,
                        row_tool,
                    )
                )

                sign_multiplier = 1.0

                # If only the reverse comparison exists,
                # use it and reverse the sign.
                if record is None:
                    record = lookup.get(
                        (
                            row_tool,
                            column_tool,
                        )
                    )

                    sign_multiplier = -1.0

                if record is None:
                    ax.text(
                        x_position,
                        y_position,
                        "NA",
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="black",
                    )

                    continue

                rank_biserial = (
                    float(record.rank_biserial)
                    * sign_multiplier
                )

                magnitude = (
                    str(record.magnitude)
                    .strip()
                    .lower()
                )

                # Background represents magnitude only.
                ax.add_patch(
                    Rectangle(
                        (
                            column_index + 0.055,
                            row_index + 0.055,
                        ),
                        0.89,
                        0.89,
                        facecolor=magnitude_background(
                            magnitude
                        ),
                        edgecolor="none",
                        linewidth=0,
                        zorder=1,
                    )
                )

                value_text = format_rank_biserial(
                    value=rank_biserial,
                    decimals=decimals,
                )

                if show_magnitude:
                    if USE_LATEX:
                        value_text += (
                            rf"\newline "
                            rf"\scriptsize{{({magnitude})}}"
                        )
                    else:
                        value_text += (
                            f"\n({magnitude})"
                        )

                ax.text(
                    x_position,
                    y_position,
                    value_text,
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="normal",
                    color="black",
                    zorder=2,
                )

        # ----------------------------------------------------
        # MATRIX BRACKETS
        # ----------------------------------------------------

        draw_bracket(
            ax=ax,
            x=-0.03,
            y_bottom=0.12,
            y_top=number_of_tools - 0.12,
            side="left",
        )

        draw_bracket(
            ax=ax,
            x=number_of_tools + 0.03,
            y_bottom=0.12,
            y_top=number_of_tools - 0.12,
            side="right",
        )

        # ----------------------------------------------------
        # SAVE PDF
        # ----------------------------------------------------

        metric_filename = (
            f"rank_biserial_"
            f"{safe_filename(metric)}.pdf"
        )

        metric_output_path = (
            output_directory
            / metric_filename
        )

        fig.savefig(
            metric_output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.01,
            transparent=False,
        )

        plt.close(fig)

        generated_files.append(
            metric_output_path
        )

        print(
            f"Created: {metric_output_path}"
        )

    return generated_files




In [25]:
# ============================================================
# EXECUTION
# ============================================================

generated_files = generate_rank_biserial_pdfs(
    csv_path="biserial_50.csv",

    output_directory=(
        "rank_biserial_matrices"
    ),

    metric_order=(
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ),

    tool_order=(
        "autogen",
        "autogpt",
        "dify",
        "semantic_kernel",
        "semantic_kernel_chat",
    ),

    decimals=3,

    # Change to True to display the magnitude label,
    # such as "(small)", under each value.
    show_magnitude=False,
)

print("\nGenerated files:")

for file_path in generated_files:
    print(file_path)

Created: rank_biserial_matrices/rank_biserial_ROUGE-1.pdf
Created: rank_biserial_matrices/rank_biserial_ROUGE-2.pdf
Created: rank_biserial_matrices/rank_biserial_ROUGE-L.pdf

Generated files:
rank_biserial_matrices/rank_biserial_ROUGE-1.pdf
rank_biserial_matrices/rank_biserial_ROUGE-2.pdf
rank_biserial_matrices/rank_biserial_ROUGE-L.pdf


In [26]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def create_effect_size_legend(
    output="effect_size_legend.pdf",
):
    """
    Create a standalone legend for effect-size magnitude.
    """

    colors = {
        "Negligible": "#D9EAD3",
        "Small": "#FFF2CC",
        "Medium": "#FCE5CD",
        "Large": "#F4CCCC",
    }

    fig, ax = plt.subplots(figsize=(6.2, 0.9))

    ax.set_xlim(0, 8)
    ax.set_ylim(0, 1)
    ax.axis("off")

    x = 0.2

    for label, color in colors.items():

        ax.add_patch(
            Rectangle(
                (x, 0.35),
                0.35,
                0.35,
                facecolor=color,
                edgecolor="black",
                linewidth=0.4,
            )
        )

        ax.text(
            x + 0.45,
            0.525,
            label,
            va="center",
            ha="left",
            fontsize=10,
            family="serif",
        )

        x += 1.9

    fig.savefig(
        output,
        bbox_inches="tight",
        pad_inches=0.02,
    )

    plt.close(fig)

    print(f"Saved {output}")


if __name__ == "__main__":
    create_effect_size_legend()

Saved effect_size_legend.pdf
